# Title

### Imports

In [96]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics import f1_score, cohen_kappa_score, matthews_corrcoef

### Data

In [97]:
# Load the data
df_sources = pd.read_excel('data/sources_amended.xlsx')
df_centroids = pd.read_excel('data/centroids.xlsx')

# Remove other (noise) category
include_other = False
if not include_other:
    df_sources = df_sources[df_sources["code"] != "O"]
    df_centroids = df_centroids[df_centroids["code"] != "O"]

### Embbed data

In [ ]:
model_name = 'intfloat/multilingual-e5-large-instruct'
prompt = "Instruct: Retrieve semantically similar text \nQuery: "

model = SentenceTransformer(model_name, 
                            local_files_only=False,
                            ) 

In [ ]:
## Vectorize the text
embeddings_sources = model.encode(df_sources["Response"].values.tolist(), 
                                  prompt=prompt, 
                                  normalize_embeddings=True)
df_sources["embeddings"] = embeddings_sources.tolist()

embeddings_centroids = model.encode(df_centroids["text"].values.tolist(), 
                                    prompt=prompt, 
                                    normalize_embeddings=True)
df_centroids["embeddings"] = embeddings_centroids.tolist()

In [ ]:
def create_centroids(df: pd.DataFrame,
                     code_col: str = "code",
                     emb_col: str = "embeddings",) -> tuple[np.ndarray, list]:
    """
    Compute one centroid per code by averaging row embeddings.

    Returns
    -------
    centroids : (C, D) array
        L2-normalized centroids stacked in the same order as `labels`.
    labels : list
        Sorted unique codes corresponding to rows in `centroids`.
    """
    if code_col not in df or emb_col not in df:
        raise KeyError(f"DataFrame must contain '{code_col}' and '{emb_col}' columns.")

    # Ensure deterministic order
    labels = sorted(df[code_col].unique().tolist())

    centroids: list[np.ndarray] = []
    for code in labels:
        # Stack all embeddings for this code
        embs = df.loc[df[code_col] == code, emb_col]
        # Support either arrays in .values or in Python lists
        stacked = np.vstack(embs)
        centroids.append(stacked.mean(axis=0))

    centroids_arr = np.asarray(centroids)

    return centroids_arr, labels


def predict_codes(embeddings: np.ndarray,
                  centroids: np.ndarray,
                  labels: list) -> list:
    """
    Assign each embedding the label of its most similar centroid (cosine).

    Parameters
    ----------
    embeddings : (N, D) array
    centroids  : (C, D) array (assumed L2-normalized row-wise)
    labels     : list of length C mapping centroid index -> code label
    """

    # Normalize so cosine similarity is dot product
    embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)
    centroids /= np.linalg.norm(centroids, axis=1, keepdims=True)

    S = embeddings @ centroids.T  # (N, C) cosine similarities

    closest_arg = np.argmax(S, axis=1)
    return [labels[i] for i in closest_arg]


def compute_metrics(y_true: np.ndarray | list, y_pred: np.ndarray | list) -> dict[str, float]:
    """
    Compute agreement/quality metrics for categorical predictions.
    """
    return {
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted")),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
    }

In [ ]:
# Build centroids
centroids, labels = create_centroids(df_centroids, code_col="code", emb_col="embeddings")

# Predict codes for new embeddings (shape: N x D)
pred_codes = predict_codes(embeddings_sources, centroids, labels)

# Evaluate against ground truth in df_sources
metrics = compute_metrics(df_sources["code"].values, pred_codes)
print(f"Kappa: {metrics['kappa']:.4f}, F1: {metrics['f1_weighted']:.4f}, MCC: {metrics['mcc']:.4f}")


Kappa: 0.4553, F1: 0.7609, MCC: 0.4853


## Sampling

one way to check stability for which centroid one uses 

In [ ]:
def sample_df_centroid(df: pd.DataFrame, 
                 sampling_per: float = 0.75, 
                 code_col: str = "code") -> pd.DataFrame:
    """
    Sample a subset of codes based on the specified sampling percentage.
    """
    code_counts = df[code_col].value_counts().to_dict()
    sampled_df = pd.DataFrame(columns=df.columns)
    
    for code, count in code_counts.items():
        n_samples = int(count * sampling_per)
        sampled_df = pd.concat([sampled_df, df[df[code_col] == code].sample(n=n_samples)])
    return sampled_df.reset_index(drop=True)

In [ ]:
n_runs = 100

df_results = pd.DataFrame(columns=["kappa", "f1_weighted", "mcc"])

for i in range(n_runs):
    sampled_df = sample_df_centroid(df_centroids, sampling_per=0.75, code_col="code")
    # Build centroids for the sampled data
    centroids_sampled, labels_sampled = create_centroids(sampled_df,
                                                        code_col="code",
                                                        emb_col="embeddings")
    
    # Predict codes for the original embeddings using the sampled centroids
    pred_codes_sampled = predict_codes(embeddings_sources, centroids_sampled, labels_sampled)
    df_results.loc[i] = compute_metrics(df_sources["code"].values, pred_codes_sampled)

# Calculate the mean and standard deviation of the metrics
mean_metrics = df_results.mean()
std_metrics = df_results.std()

print(f"Mean Kappa: {mean_metrics['kappa']:.4f} ± {std_metrics['kappa']:.4f}")
print(f"Mean F1: {mean_metrics['f1_weighted']:.4f} ± {std_metrics['f1_weighted']:.4f}")
print(f"Mean MCC: {mean_metrics['mcc']:.4f} ± {std_metrics['mcc']:.4f}")


Mean Kappa: 0.4321 ± 0.0233
Mean F1: 0.7480 ± 0.0133
Mean MCC: 0.4646 ± 0.0220
